# Manual Single-File PMT Response (without first 3 layers)

Select an MC set and flavor - the notebook picks an example skimmed I3Photons file and runs PMT response on it.  
For testing a single file without submitting a batch job.  
Mirrors `apply_pmt_response_without_first_3_layers.py`.

In [6]:
MC       = "Spring2026MC"        # "Spring2026MC"  |  "340StringMC"
FLAVOR   = "Muon"                # "Muon" | "Electron" | "Tau" | "NC"
GEOMETRY = "strings_102_40m"     # sub-geometry key matching paths.py

TASK_ID  = 0                     # index of the file to pick from the sorted input list

In [7]:
import sys
import re
from pathlib import Path

PROJECT_ROOT = Path("/project/def-nahee/kbas/Graphnet-Applications")

sys.path.insert(0, str(PROJECT_ROOT / "Metadata"))
import paths

# input: skimmed I3Photons files for this geometry/flavor
if MC == "Spring2026MC":
    i3_entry  = paths.SPRING2026MC_I3[GEOMETRY][FLAVOR]
    pmt_entry = paths.SPRING2026MC_PMT[GEOMETRY][FLAVOR]
elif MC == "340StringMC":
    i3_entry  = paths.STRING340MC_I3[GEOMETRY][FLAVOR]
    pmt_entry = paths.STRING340MC_PMT[GEOMETRY][FLAVOR]
else:
    raise ValueError(f"Unknown MC: {MC}")

INDIR   = Path(i3_entry["path"])
PATTERN = "*.i3.gz"

# trimmed GCD matching the sub-geometry
GCD     = Path(paths.GCD_TRIMMED[MC][GEOMETRY])

# output directory — same path as the batch pipeline uses
OUTDIR  = Path(pmt_entry["path"])

print(f"MC      : {MC}")
print(f"FLAVOR  : {FLAVOR}")
print(f"GEOMETRY: {GEOMETRY}")
print(f"INDIR   : {INDIR}")
print(f"GCD     : {GCD}")
print(f"OUTDIR  : {OUTDIR}")

for label, p in [("INDIR", INDIR), ("GCD", GCD)]:
    status = "OK" if Path(p).exists() else "MISSING"
    print(f"  [{status}] {label}")

MC      : Spring2026MC
FLAVOR  : Muon
GEOMETRY: strings_102_40m
INDIR   : /scratch/kbas/Spring2026MC/Strings_102_40m/Muon_I3Photons
GCD     : /project/def-nahee/kbas/Graphnet-Applications/Metadata/GCD/Spring2026MC/strings_102_40m.i3.gz
OUTDIR  : /home/kbas/scratch/Spring2026MC/Strings_102_40m/Muon_PMT_Response
  [OK] INDIR
  [OK] GCD


In [8]:
# list input files and pick one by TASK_ID

all_files = sorted(INDIR.rglob(PATTERN))

print(f"Found {len(all_files)} files matching '{PATTERN}' in {INDIR}")

if not all_files:
    raise FileNotFoundError("No files found — check INDIR and PATTERN")

if not (0 <= TASK_ID < len(all_files)):
    raise IndexError(f"TASK_ID={TASK_ID} out of range (0..{len(all_files)-1})")

INFILE = all_files[TASK_ID]
print(f"\nSelected file (TASK_ID={TASK_ID}):")
print(f"  {INFILE}")

Found 1 files matching '*.i3.gz' in /scratch/kbas/Spring2026MC/Strings_102_40m/Muon_I3Photons

Selected file (TASK_ID=0):
  /scratch/kbas/Spring2026MC/Strings_102_40m/Muon_I3Photons/muon_38460126_gen_001.i3.gz


In [9]:
# extract run number from filename (used as random service stream)

m = re.search(r"gen_(\d+)", INFILE.name) or re.search(r"cls_(\d+)", INFILE.name)
if not m:
    raise ValueError(f"Could not parse run number from filename: {INFILE.name}")

RUNNUMBER = int(m.group(1))
print(f"Run number: {RUNNUMBER}")

Run number: 1


In [10]:
import os
import sys
import h5py  # must be imported before icecube to avoid HDF5 version conflict
import time
import numpy as np
from copy import deepcopy
from math import sqrt

# required by pone_offline/Utilities/DOMUtility.py at import time
if not os.environ.get("PONESRCDIR"):
    os.environ["PONESRCDIR"] = "/project/6008051/pone_simulation/pone_offline"

from icecube.icetray import I3Tray, I3Units, I3Frame, OMKey
from icecube import icetray, dataclasses, dataio, simclasses
from icecube import phys_services, sim_services
from icecube.dataclasses import ModuleKey

from DOM.PONEDOMLauncher import DOMSimulation
from DOM.OMAcceptance import OMAcceptance
from NoiseGenerators.DarkNoise import DarkNoise
from NoiseGenerators.K40Noise import K40Noise
from Trigger.DOMTrigger import DOMTrigger
from Trigger.DetectorTrigger import DetectorTrigger


class HitCountCheck(icetray.I3Module):
    def __init__(self, context):
        super().__init__(context)
        self.AddParameter("NHits", "Minimum number of unique OMs required to pass frame", 5)

    def Configure(self):
        self.NHits = self.GetParameter("NHits")

    def DAQ(self, frame):
        unique_oms = set((k.string, k.om) for k in frame["PMT_Response"].keys())
        if len(unique_oms) < self.NHits:
            return False
        self.PushFrame(frame)


class FixTriggerMap(icetray.I3Module):
    def __init__(self, context):
        super().__init__(context)
        self.AddOutBox("OutBox")

    def Configure(self):
        pass

    def DAQ(self, frame):
        bad_map = frame["triggerpulsemap"]
        dom_map = dataclasses.I3RecoPulseSeriesMap()
        for omkey in bad_map.keys():
            pmt     = omkey.pmt
            dom_key = OMKey(omkey.string, omkey.om, 0)
            if dom_key not in dom_map.keys():
                dom_map[dom_key] = dataclasses.I3RecoPulseSeries()
            for pulse in bad_map[omkey]:
                new_pulse        = dataclasses.I3RecoPulse()
                new_pulse.time   = pulse.time
                new_pulse.charge = pulse.charge
                new_pulse.width  = float(pmt)
                dom_map[dom_key].append(new_pulse)
        for dom_key in dom_map.keys():
            pulses = sorted(dom_map[dom_key], key=lambda p: p.time)
            series = dataclasses.I3RecoPulseSeries()
            for p in pulses:
                series.append(p)
            dom_map[dom_key] = series
        frame.Delete("triggerpulsemap")
        frame["triggerpulsemap"] = dom_map
        self.PushFrame(frame)


print("All imports OK")


All imports OK


In [11]:
# resolve output path (same naming convention as the batch script)

OUTDIR.mkdir(parents=True, exist_ok=True)

rel      = INFILE.relative_to(INDIR)
parts    = list(rel.parts)
suffixes = "".join(INFILE.suffixes)
parts[-1] = parts[-1][:-len(suffixes)] if suffixes else INFILE.stem
stem     = "_".join(parts)

OUTFILE = OUTDIR / f"{FLAVOR.lower()}_{stem}.i3.gz"

print(f"Output file: {OUTFILE}")

Output file: /home/kbas/scratch/Spring2026MC/Strings_102_40m/Muon_PMT_Response/muon_muon_38460126_gen_001.i3.gz


In [13]:
# run PMT response

pulsesep = 0.2
nDOMs    = 1

randomService = phys_services.I3SPRNGRandomService(
    seed=1234567, nstreams=10000000, streamnum=RUNNUMBER
)

print(f"Processing: {INFILE.name}")
print(f"-> {OUTFILE}")

t0 = time.time()

tray = I3Tray()
tray.context["I3RandomService"] = randomService
tray.AddModule("I3Reader", "reader", FilenameList=[str(GCD), str(INFILE)])

tray.AddModule(
    DOMSimulation, "DOMLauncher",
    input_map="Accepted_MCPEMap",
    output_map="PMT_Response",
    random_service=randomService,
    min_time_sep=pulsesep,
    split_doms=True,
    use_dark=True, dark_map="Noise_Dark",
    use_k40=True,  k40_map="Noise_K40",
)

tray.AddModule(HitCountCheck, "hitcheck", NHits=5)

tray.AddModule(FixTriggerMap, "FixTriggerMap")

tray.AddModule(DOMTrigger, "DOMTrigger", trigger_map="triggerpulsemap")

tray.AddModule(
    DetectorTrigger, "PONE_Trigger",
    output="_3PMT_1DOM",
    OMPMTCoinc=3,
    FullDetectorCoincidenceN=nDOMs,
    CutOnTrigger=True,
    EventLength=10000,
    TriggerTime=2000,
    PulseSeriesIn="PMT_Response",
    PulseSeriesOut="EventPulseSeries",
)

tray.AddModule(
    DetectorTrigger, "PONE_Trigger_nonoise",
    output="_3PMT_1DOM_nonoise",
    OMPMTCoinc=3,
    FullDetectorCoincidenceN=nDOMs,
    CutOnTrigger=True,
    EventLength=10000,
    TriggerTime=2000,
    PulseSeriesIn="PMT_Response_nonoise",
    PulseSeriesOut="EventPulseSeries_nonoise",
)

tray.AddModule(
    "I3Writer", "writer",
    Filename=str(OUTFILE),
    Streams=[icetray.I3Frame.DAQ, icetray.I3Frame.Simulation],
)

tray.Execute()
tray.Finish()

elapsed = time.time() - t0
size_mb = OUTFILE.stat().st_size / 1e6
print(f"Done in {elapsed:.1f}s — output size: {size_mb:.1f} MB")

Processing: muon_38460126_gen_001.i3.gz
-> /home/kbas/scratch/Spring2026MC/Strings_102_40m/Muon_PMT_Response/muon_muon_38460126_gen_001.i3.gz


: 